# 实验六：MSPROF 性能 Profiling 与瓶颈定位

本章使用 MSPROF 分析 `1*NPU 910B3` 上的 YOLO 训练。当前没有多卡通信，因此 profiling 重点不是 HCCL wait，而是 DataLoader gap、Host 调度、NPU kernel 耗时、AICPU 算子和不必要同步点。

`Profiling` 的意思是性能剖析：把训练过程拆成一段段时间线，观察时间花在哪里。MSPROF 是 Ascend 平台常用的 profiling 工具。


## 采集命令

在实验目录执行：

```bash
bash src/scripts/profile_msprof.sh
```

脚本会把训练缩短到少量 epoch，并把 profiling 结果写到：

```text
profiles/yolo_single_npu/
```

采集完成后运行：

```bash
python src/scripts/analyze_msprof.py --profile-dir profiles/yolo_single_npu
```


In [ ]:
# ====== 1. 查看 profiling 脚本 ======
from pathlib import Path

print(Path('src/scripts/profile_msprof.sh').read_text(encoding='utf-8'))


## 第一次看 MSPROF 时关注什么

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">现象</th>
      <th style="text-align: left;">可能原因</th>
      <th style="text-align: left;">优先尝试</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">step 之间有很长空白</td>
      <td style="text-align: left;">DataLoader 供数慢</td>
      <td style="text-align: left;">增大 <code>workers</code>、检查数据盘、减少在线增强</td>
    </tr>
    <tr>
      <td style="text-align: left;">Host runtime API 很密集</td>
      <td style="text-align: left;">Python 侧同步或小操作太多</td>
      <td style="text-align: left;">减少频繁 <code>.item()</code>、合并日志打印</td>
    </tr>
    <tr>
      <td style="text-align: left;">AICPU 算子耗时突出</td>
      <td style="text-align: left;">部分算子落到 CPU 或 AICPU</td>
      <td style="text-align: left;">检查算子支持、替换不友好操作</td>
    </tr>
    <tr>
      <td style="text-align: left;">AI Core kernel 时间短但间隔大</td>
      <td style="text-align: left;">NPU 没被持续喂满</td>
      <td style="text-align: left;">增大 batch size 或优化数据读取</td>
    </tr>
    <tr>
      <td style="text-align: left;">loss 变 NaN</td>
      <td style="text-align: left;">学习率、AMP 或 label 异常</td>
      <td style="text-align: left;">降学习率、增加 warmup、检查 VOC label 转换</td>
    </tr>
  </tbody>
</table>


In [ ]:
# ====== 2. 生成瓶颈定位建议 ======
def suggest(symptom):
    table = {
        'data_gap': ['workers 从 4/8/12 逐步试验', '确认数据在 /mnt/workspace', '减少过重在线增强'],
        'oom': ['batch_size 减半', '保持 AMP=true', '必要时使用 gradient_accumulation'],
        'nan_loss': ['学习率减半', 'warmup_epochs 增加到 5', '检查 VOC label 转换是否正确'],
        'host_busy': ['减少过密日志', '避免每 step 调用太多同步操作', '把统计汇总到 log_interval'],
    }
    return table.get(symptom, ['先查看 MSPROF timeline，再定位耗时最长的阶段'])

for symptom in ['data_gap', 'oom', 'nan_loss', 'host_busy']:
    print(symptom, '->')
    for item in suggest(symptom):
        print('  -', item)


## 如何形成实验结论

建议把每次实验记录成下面这种表格：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">batch_size</th>
      <th style="text-align: left;">workers</th>
      <th style="text-align: left;">AMP</th>
      <th style="text-align: left;">平均 imgs/s</th>
      <th style="text-align: left;">MSPROF 主要瓶颈</th>
      <th style="text-align: left;">结论</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">8</td>
      <td style="text-align: left;">4</td>
      <td style="text-align: left;">on</td>
      <td style="text-align: left;"></td>
      <td style="text-align: left;"></td>
      <td style="text-align: left;"></td>
    </tr>
    <tr>
      <td style="text-align: left;">16</td>
      <td style="text-align: left;">8</td>
      <td style="text-align: left;">on</td>
      <td style="text-align: left;"></td>
      <td style="text-align: left;"></td>
      <td style="text-align: left;"></td>
    </tr>
    <tr>
      <td style="text-align: left;">24</td>
      <td style="text-align: left;">8</td>
      <td style="text-align: left;">on</td>
      <td style="text-align: left;"></td>
      <td style="text-align: left;"></td>
      <td style="text-align: left;"></td>
    </tr>
    <tr>
      <td style="text-align: left;">16</td>
      <td style="text-align: left;">12</td>
      <td style="text-align: left;">on</td>
      <td style="text-align: left;"></td>
      <td style="text-align: left;"></td>
      <td style="text-align: left;"></td>
    </tr>
  </tbody>
</table>

最终结论不要只写“更快了”，而要说明为什么更快：是 batch size 提高了 NPU 利用率，还是 workers 减少了 DataLoader gap，或者 AMP 降低了显存与计算压力。


## 全实验总结

到这里，你已经完成了当前硬件条件下的大规模训练加速与调优闭环：

1. 选择 PASCAL VOC 作为适合当前资源的目标检测训练数据集。
2. 将 VOC XML 标注转换为 YOLO txt，并生成 train/val split。
3. 在单卡 Ascend 910B3 NPU 上启动 YOLO 训练。
4. 使用 AMP、Warmup、batch size 和 DataLoader workers 做稳定性与吞吐调优。
5. 使用 MSPROF 分析 DataLoader、Host 和 NPU 计算瓶颈。

等具备多卡资源后，可以复用同一训练脚本启用 HCCL/DDP，再补充分布式吞吐和扩展效率实验。


## 课后练习

请根据本节实验内容完成以下练习。题型包含单选题、多选题、判断题、填空题、简答题和代码设计题。

1. (单选题) MSPROF 在本实验中的主要用途是？
   - A. 采集训练过程性能数据并辅助定位瓶颈
   - B. 转换 XML 标注
   - C. 生成 VOC 数据集
   - D. 替代优化器

2. (单选题) Profiling 结果中，`Task Duration(us)` 更接近表示什么？
   - A. 算子或任务耗时
   - B. 图片数量
   - C. 类别编号
   - D. Git 提交数量

3. (单选题) 如果 runner 端到端耗时远大于单个算子耗时，最合理的解释是？
   - A. 端到端时间包含数据生成、编译、初始化、文件 I/O 等开销
   - B. NPU 一定比 CPU 慢
   - C. 模型没有类别
   - D. VOC XML 无法读取

4. (单选题) 生成 Profiling 数据后，实验报告中应优先记录什么？
   - A. 数据目录、关键 csv、目标算子耗时和结论
   - B. 屏幕亮度
   - C. 鼠标型号
   - D. 浏览器缩放比例

5. (多选题) MSPROF 采集后可能关注哪些文件或信息？
   - A. op_summary_*.csv
   - B. task_time_*.csv
   - C. api_statistic_*.csv
   - D. mindstudio_profiler_output 目录

6. (多选题) 分析训练瓶颈时，应该同时考虑哪些方面？
   - A. Host 侧数据加载
   - B. NPU 计算耗时
   - C. 内存拷贝和同步
   - D. 日志和 checkpoint I/O

7. (多选题) 一个严谨的性能结论应避免哪些说法？
   - A. 只凭一次运行下结论
   - B. 混淆端到端耗时和算子耗时
   - C. 只报最快值不说明环境
   - D. 说明采集命令和数据位置

8. (判断题) Profiling 的目标是证明程序一定最快，而不是定位瓶颈。

9. (判断题) 同一实验中 CPU baseline、训练日志和 MSPROF 数据可以互相补充。

10. (填空题) MSPROF 生成的数据目录常以 `____` 开头。

11. (填空题) 分析算子耗时时，常在 `op_summary_*.csv` 中查找目标算子的 `____` 字段。

12. (简答题) 为什么性能分析要区分“端到端耗时”和“算子耗时”？

13. (简答题) 如果 Profiling 显示 NPU 计算很快但整体训练慢，下一步应该看什么？

14. (简答题) 实验验收时如何描述 MSPROF 的分析结果更严谨？

15. (代码设计题) 写一条示例命令，用 MSPROF 采集 2 个 epoch 的单卡训练性能数据。

> 参考答案见 answer/03.07_msprof_profiling_and_chapter_test_answer.ipynb。
